In this notebook we are going to compute different connectivity measures and aggregate them into a dataframe. 

In [ ]:
import numpy as np
import mne

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import ttest_rel
import pandas as pd
import os 
from tqdm import tqdm
from IPython.display import clear_output
import warnings 
from mne_connectivity import seed_target_indices

# I only want to see warnings once
warnings.filterwarnings('ignore')


Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')

In [ ]:
from LFPAnalysis import oscillation_utils


## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

# Encoding connectivity:

In [ ]:
# Explicitly define a list of analysis events and the baseline event. Should correspond to the dict

evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

for i in analysis_evs: 
    if i not in evs.keys(): 
        raise ValueError('This analysis event is not in the dictionary')

## Directed Connectivity (PSI or Granger) - computed over epochs

In [ ]:
analysis_evs = ['feedback_start']

In [ ]:
metric = 'psi'
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

# freq_dict = {'theta':[4, 8],
#             'alpha':[8, 13],
#             'beta': [13, 30], 
#             'hfa': [70, 200]}

freq_dict = {'high_theta':[4, 8]}

band1 = 'high_theta'
band2 = 'high_theta'

# band1 = 'theta'
# band2 = 'theta'

buf_ms = 1000
n_surr = 500

savepath = f'{base_dir}/scratch/MemoryBandit/{metric}'
if not os.path.exists(savepath):
    os.makedirs(savepath)

In [ ]:
def make_seed_target_df(elec_df, epochs, source_roi, target_roi): 
    
    """
    Create arrays of indices for mapping electrodes for connectivity analyses. Use the Epoch
    ch_names list itself to find the index of the electrode within the mne object 
    
    """
    
    seed_target_df = pd.DataFrame(columns=['seed', 'target'], index=['l', 'r'])

    for hemi in ['l', 'r']:
        source_ix = elec_df[(elec_df.hemisphere.str.lower()==hemi) & (elec_df.salman_region==source_roi)].label.values
        target_ix = elec_df[(elec_df.hemisphere.str.lower()==hemi) & (elec_df.salman_region==target_roi)].label.values

        if (len(source_ix) == 0) | (len(target_ix)==0):
            seed_target_df['seed'][hemi] = []
            seed_target_df['target'][hemi] = []
        else:   
            seed_target_df['seed'][hemi] = mne.pick_channels(epochs.ch_names, source_ix)

            seed_target_df['target'][hemi] = mne.pick_channels(epochs.ch_names, target_ix)

    seed_target_df = seed_target_df[
                (seed_target_df['seed'].map(lambda d: len(d) > 0)) & (seed_target_df['target'].map(lambda d: len(d) > 0))]

    return seed_target_df
    

In [ ]:
# %%capture

# # pairs = (['AMY', 'HPC'], ['AMY', 'OFC'], ['AMY', 'dmPFC'], ['AMY', 'ACC'])
# # ['HPC', 'OFC'], , ['ACC', 'OFC'], ['dmPFC','OFC'], ['dlPFC', 'OFC']
# pairs = (['dmPFC', 'OFC'],)

# for pair in pairs:
#     source_region = pair[0]
#     target_region = pair[1]
    
#     # initialize the storage list 
#     conn_group_data = []
#     conn_group_metadata = []
    
#     # iterate through subjects
#     for subj_id in subj_ids:
#         for event in analysis_evs:
#             loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
#             epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True, verbose='ERROR'); 
            
#             # MEMORY BASED PARSING
#             memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
#             memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
#             memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else 0, axis=1)
            
#             # hit_index = memory_df[memory_df.hits==True].trials.values.astype(int) - 1
#             # trunc_epochs = tfr_data[epochs_to_analyze.astype(int)]

            
#             # # RPE BASED PARSING
#             # pos_rpe_index = np.where(learn_df[learn_df.participant==subj_id].rpe>0)[0]           

#             # Get electrode df 
#             elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

#             seed_target_df = make_seed_target_df(elec_df, epochs_reref, source_region, target_region)
            
#             if len(seed_target_df) == 0:
#                 continue
                
#             # for cond in conditions: 
#             for hemi in ['l', 'r']:
#                 # first determine if ipsilateral connectivity is even possible; if not, move on. 
#                 # note: edit this if you want to compute contralateral connectivity 
#                 if hemi not in seed_target_df.index.tolist():
#                     continue
#                 else:
#                     seed_to_target = seed_target_indices(
#                         seed_target_df['seed'][hemi],
#                         seed_target_df['target'][hemi])
            
            
#                 pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
#                                                band = freq_dict[band1], 
#                                                metric = metric, 
#                                                indices = seed_to_target, 
#                                                freqs = freqs, 
#                                                n_cycles = n_cycles,
#                                                buf_ms = buf_ms, 
#                                                n_surr=n_surr,
#                                                avg_over_dim='epochs',
#                                                band1 = freq_dict[band2],
#                                                parallelize=True,
#                                                gc_n_lags=20)

#                 npairs = len(seed_to_target[0])
#                 pair_labels = [] 
#                 for pair_ix in range(npairs):
#                     pair_labels.append(f'{epochs_reref.ch_names[seed_to_target[0][pair_ix]]}:{epochs_reref.ch_names[seed_to_target[1][pair_ix]]}')


#                 row_metadata = pd.DataFrame({
#                     'pair_labels': pair_labels,
#                     'participant': subj_id,
#                     'age': subj_df[subj_df.MSSMCode==subj_id].Age.values[0], 
#                     'sex': subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0],
#                     'roi1': source_region, 
#                     'roi2': target_region, 
#                     'hemi': hemi, 
#                     'metric': metric,
#                     'event': event,
#                     'band0': band1, 
#                     'band1': band2,
#                     'cond':'hit'}) # ACTION: SWAP

#                 conn_group_data.append(pwise)
#                 conn_group_metadata.append(row_metadata)
            
#     conn_group_data = np.vstack(conn_group_data)
#     conn_group_metadata = pd.concat(conn_group_metadata)
    
#     # ACTION: SWAP
#     np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_Z.npy', conn_group_data)
#     conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df_Z', index=False)
    

Separate out the conditions we want to analyze using PSI. Previously I did positive vs. negative rPE

In [ ]:
# pairs = (['AMY', 'HPC'], ['AMY', 'OFC'], ['AMY', 'dmPFC'], ['AMY', 'ACC'])
# ['HPC', 'OFC'], , ['ACC', 'OFC'], ['dmPFC','OFC'], ['dlPFC', 'OFC']
pairs = (['HPC', 'OFC'], )

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []
    conn_group_metadata = []
    
    # iterate through subjects
    for subj_id in tqdm(subj_ids, desc=f'{source_region} to {target_region} connectivity using {metric}'): 
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True, verbose='ERROR'); 
            
            # MEMORY BASED PARSING
            if event == 'feedback_start':
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
                memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
                memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else 0, axis=1)

                hit_index = memory_df[memory_df.hits==True].trials.values.astype(int) - 1
            elif event == 'cue_on':
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_mem']).sort_values(by=['trials_mem'])[['participant', 'trials_mem', 'hits']]
                memory_df.rename(columns={'trials_mem':'trials'}, inplace=True)
                memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else 0, axis=1)

                hit_index = memory_df[memory_df.hits==True].trials.values.astype(int) - 1
            # trunc_epochs = tfr_data[epochs_to_analyze.astype(int)]

            
            # # RPE BASED PARSING
            # pos_rpe_index = np.where(learn_df[learn_df.participant==subj_id].rpe>0)[0]           

            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)
            
            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
            
            
                pwise = oscillation_utils.compute_connectivity(epochs_reref.copy()[hit_index], 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='epochs',
                                               surr_method='swap_epochs',
                                               band1 = freq_dict[band2],
                                               parallelize=True,
                                               gc_n_lags=20)

                npairs = len(seed_to_target[0])
                pair_labels = [] 
                for pair_ix in range(npairs):
                    pair_labels.append(f'{epochs_reref.ch_names[seed_to_target[0][pair_ix]]}:{epochs_reref.ch_names[seed_to_target[1][pair_ix]]}')


                row_metadata = pd.DataFrame({
                    'pair_labels': pair_labels,
                    'participant': subj_id,
                    'age': subj_df[subj_df.MSSMCode==subj_id].Age.values[0], 
                    'sex': subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0],
                    'roi1': source_region, 
                    'roi2': target_region, 
                    'hemi': hemi, 
                    'metric': metric,
                    'event': event,
                    'band0': band1, 
                    'band1': band2,
                    'cond':'hit'}) # ACTION: SWAP

                conn_group_data.append(pwise)
                conn_group_metadata.append(row_metadata)
            
    conn_group_data = np.vstack(conn_group_data)
    conn_group_metadata = pd.concat(conn_group_metadata)
    
    # ACTION: SWAP
    np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z_swap_epochs.npy', conn_group_data)
    conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z_swap_epochs_df', index=False)
    

In [ ]:
%%capture

# pairs = (['AMY', 'HPC'], ['AMY', 'OFC'], ['AMY', 'dmPFC'], ['AMY', 'ACC'])
# ['HPC', 'OFC'], , ['ACC', 'OFC'], ['dmPFC','OFC'], ['dlPFC', 'OFC']
pairs = (['HPC', 'OFC'], )

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []
    conn_group_metadata = []
    
    # iterate through subjects
    for subj_id in subj_ids:
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True); 
            
            # MEMORY BASED PARSING
            if event == 'feedback_start':
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
                memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
                memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else 0, axis=1)

                miss_index = memory_df[memory_df.hits==False].trials.values.astype(int) - 1
            elif event == 'cue_on':
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_mem']).sort_values(by=['trials_mem'])[['participant', 'trials_mem', 'hits']]
                memory_df.rename(columns={'trials_mem':'trials'}, inplace=True)
                memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else 0, axis=1)

                miss_index = memory_df[memory_df.hits==False].trials.values.astype(int) - 1
            # trunc_epochs = tfr_data[epochs_to_analyze.astype(int)]

            
            # # RPE BASED PARSING
            # pos_rpe_index = np.where(learn_df[learn_df.participant==subj_id].rpe>0)[0]           

            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)

            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
            
            
                pwise = oscillation_utils.compute_connectivity(epochs_reref.copy()[miss_index], # ACTION: SWAP
                                                   band = freq_dict[band1], 
                                                   metric = metric, 
                                                   indices = seed_to_target, 
                                                   freqs = freqs, 
                                                   n_cycles = n_cycles,
                                                   buf_ms = buf_ms, 
                                                   n_surr=n_surr,
                                                   avg_over_dim='epochs',
                                                   surr_method = 'swap_epochs',
                                                   band1 = freq_dict[band2],
                                                   parallelize=True)

                npairs = len(seed_to_target[0])
                pair_labels = [] 
                for pair_ix in range(npairs):
                    pair_labels.append(f'{epochs_reref.ch_names[seed_to_target[0][pair_ix]]}:{epochs_reref.ch_names[seed_to_target[1][pair_ix]]}')


                row_metadata = pd.DataFrame({
                    'pair_labels': pair_labels,
                    'participant': subj_id,
                    'age': subj_df[subj_df.MSSMCode==subj_id].Age.values[0], 
                    'sex': subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0],
                    'roi1': source_region, 
                    'roi2': target_region, 
                    'hemi': hemi, 
                    'metric': metric,
                    'event': event,
                    'band0': band1, 
                    'band1': band2,
                    'cond':'miss'}) # ACTION: SWAP

                conn_group_data.append(pwise)
                conn_group_metadata.append(row_metadata)
            
    conn_group_data = np.vstack(conn_group_data)
    conn_group_metadata = pd.concat(conn_group_metadata)
    
    # ACTION: SWAP
    np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z_swap_epochs.npy', conn_group_data)
    conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z_swap_epochs_df', index=False)
    

In [ ]:
source_region = 'HPC'

band1= 'high_theta'
band2= 'high_theta'
event = 'feedback_start'
metric = 'psi'

for target_region in ['OFC']:
    conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z_swap_epochs.npy')
    conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z_swap_epochs.npy')

    fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)

    # meta_data = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z_df')
    # meta_data.hemi = meta_data.hemi.str.lower()
    # hemi_index = meta_data[meta_data.hemi=='r'].index.values
    # conn_group_dat = conn_group_data_pos - conn_group_data_neg

    T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test([conn_group_data_pos[:, :], conn_group_data_neg[:, :]], 
                                                                                     n_permutations=1000,
                                                                                     out_type='mask', seed=1)
    # T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_1samp_test(conn_group_data_neg, 
    #                                                                                  n_permutations=1000,
    #                                                                                  out_type='mask')
    print(cluster_p_values)
    stderr = np.nanstd(conn_group_data_pos[:, :], axis=0) / np.sqrt(conn_group_data_pos[:, :].shape[0])
    mean_data = np.nanmean(conn_group_data_pos[:, :], axis=0)

    freqs = np.linspace(0, 1500, conn_group_data_pos[:, :].shape[1])
    # freqs = np.linspace(0, 500, conn_group_data_pos[:, :].shape[1])

    ax.plot(freqs, mean_data)
    ax.fill_between(freqs, mean_data - stderr, mean_data + stderr, color='green', alpha=0.3)

    stderr = np.nanstd(conn_group_data_neg[:, :], axis=0) / np.sqrt(conn_group_data_neg[:, :].shape[0])
    mean_data = np.nanmean(conn_group_data_neg[:, :], axis=0)


    ax.plot(freqs, mean_data)
    ax.fill_between(freqs, mean_data - stderr, mean_data + stderr, color='red', alpha=0.3)

    ax.hlines(0, freqs[0], freqs[-1], 'k')

    for i_c, c in enumerate(clusters):
        c = c[0]
        if cluster_p_values[i_c] <= 0.05:
            if target_region=='HPC':
                h = ax.axvspan(freqs[c.start], freqs[c.stop - 1], color="purple", alpha=0.3)
            elif target_region=='AMY':
                h = ax.axvspan(freqs[c.start], freqs[c.stop - 1], color="yellow", alpha=0.3)
            else: 
                h = ax.axvspan(freqs[c.start], freqs[c.stop - 1], color="k", alpha=0.3)
#         else:
#             ax.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)


    sns.despine()
    ax.set(xlabel='time (ms)', ylabel='phase slope index (z)')
    # plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{metric}_{source_region}_{target_region}_{band1}_{event}_memory_Z.pdf', dpi=300)


In [ ]:
# grouped across subjects, pairwise granger is utterly unimpressive during both feedback and cue

In [ ]:
meta_data = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z_df')

In [ ]:
meta_data['conn'] = np.nanmean(conn_group_data_neg, axis=1)

In [ ]:
meta_data

In [ ]:
# drop one subject

source_region = 'HPC'
target_region = 'OFC'
band1= 'theta'
band2= 'theta'
event = 'feedback_start'
metric = 'psi'

fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)
for target_region in ['OFC']:
    conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z.npy')
    conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z.npy')

    for subj in meta_data.participant.unique():
        fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)

        subj_index = meta_data[meta_data.participant==subj].index.values

        T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test([np.delete(conn_group_data_pos, subj_index, axis=0),
                                                                                    np.delete(conn_group_data_neg, subj_index, axis=0)], 
                                                                                         n_permutations=1000,
                                                                                         out_type='mask')
        print(cluster_p_values)
        stderr = np.nanstd(np.delete(conn_group_data_pos, subj_index, axis=0), axis=0) / np.sqrt(np.delete(conn_group_data_pos, subj_index, axis=0).shape[0])
        mean_data = np.nanmean(np.delete(conn_group_data_pos, subj_index, axis=0), axis=0)

        freqs = np.linspace(0, 1500, conn_group_data_pos[:, :].shape[1])

        ax.plot(freqs, mean_data)
        ax.fill_between(freqs, mean_data - stderr, mean_data + stderr, color='green', alpha=0.3)

        stderr = np.nanstd(np.delete(conn_group_data_neg, subj_index, axis=0), axis=0) / np.sqrt(np.delete(conn_group_data_neg, subj_index, axis=0).shape[0])
        mean_data = np.nanmean(np.delete(conn_group_data_neg, subj_index, axis=0), axis=0)


        ax.plot(freqs, mean_data)
        ax.fill_between(freqs, mean_data - stderr, mean_data + stderr, color='red', alpha=0.3)

        ax.hlines(0, 2, 8, 'k')

        for i_c, c in enumerate(clusters):
            c = c[0]
            if cluster_p_values[i_c] <= 0.05:
                if target_region=='HPC':
                    h = ax.axvspan(freqs[c.start], freqs[c.stop - 1], color="purple", alpha=0.3)
                elif target_region=='AMY':
                    h = ax.axvspan(freqs[c.start], freqs[c.stop - 1], color="yellow", alpha=0.3)
                else: 
                    h = ax.axvspan(freqs[c.start], freqs[c.stop - 1], color="k", alpha=0.3)
    #         else:
    #             ax.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)


        sns.despine()
        ax.set(xlabel='freqs (Hz)', ylabel='theta PSI')
        plt.savefig(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{metric}_{source_region}_{target_region}_{band1}_memory_Z.pdf', dpi=300)


In [ ]:
np.delete(conn_group_data_pos, subj_index, axis=0).shape


In [ ]:
subj_index

In [ ]:
ttest_rel(conn_group_data_pos.mean(axis=1), 
          conn_group_data_neg.mean(axis=1))

In [ ]:
conn_group_data_neg.mean()

In [ ]:
clusters

In [ ]:
cluster_p_values

In [ ]:
conn_group_metadata

In [ ]:
# for target_region in ['HPC', 'OFC', 'dmPFC']:
#     conn_group_data = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}.npy')
#     plt.plot(np.nanmean(conn_group_data, axis=0))
    

In [ ]:
epochs_reref._data.shape

In [ ]:
seed_to_target.shape

In [ ]:
conn_group_data.shape

In [ ]:
conn_group_metadata.shape

In [ ]:
psi = np.squeeze(test_data.get_data()[:, 0])[:, buf_rs:-buf_rs]

In [ ]:
plt.plot(np.nanmean(psi, axis=0))

In [ ]:
pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=0,
                                               avg_over_dim='epochs',
                                               band1 = freq_dict[band2],
                                               parallelize=True)

In [ ]:
seed_to_target

In [ ]:
plt.plot(np.nanmean(pwise, axis=0))

## Undirected connectivity (e.g. coh) - computed over timepoints

Let's start here because we don't need to step out of HFA then 

In [ ]:
analysis_evs = ['feedback_start']

In [ ]:
metric = 'wpli'
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

# freq_dict = {'theta':[2, 8], 
#             'alpha':[8, 13],
#             'beta': [13, 30], 
#             'hfa': [70, 200]}

freq_dict = {'high_theta':[4, 8]}

band1 = 'high_theta'
band2 = 'high_theta'

buf_ms = 1000
n_surr = 500

savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'
if not os.path.exists(savepath):
    os.makedirs(savepath)

In [ ]:
# %%capture
# from IPython.display import display

pairs = (['HPC', 'OFC'],)
# , ['dlPFC', 'OFC'], ['HPC', 'OFC'], ['dmPFC', 'OFC']
# pairs = ['AMY', 'HPC'], 
# pairs = (['HPC', 'OFC'], )

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []

    # iterate through subjects
    for subj_id in tqdm(subj_ids[4:], desc=f'{source_region} to {target_region} connectivity using {metric}'):
        # display(f'Computing granger causality for {pair} in {subj_id}')
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True) 

            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, source_region, target_region)
            
            # For within-region connectivity, let's drop elements that are the same
            
            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
                
                if metric == 'amp':
                    pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = None, 
                                               n_cycles = None,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='time',
                                               band1 = freq_dict[band2]);
                    
                elif metric == 'granger': 
                    pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='time',
                                               band1 = freq_dict[band2],
                                               parallelize=True,
                                               gc_n_lags=9);
                    
                else:
                    pwise = oscillation_utils.compute_connectivity(epochs_reref.copy(), 
                                               band = freq_dict[band1], 
                                               metric = metric, 
                                               indices = seed_to_target, 
                                               freqs = freqs, 
                                               n_cycles = n_cycles,
                                               buf_ms = buf_ms, 
                                               n_surr=n_surr,
                                               avg_over_dim='time',
                                               surr_method = 'swap_epochs',
                                               band1 = freq_dict[band2],
                                               parallelize=True)

                pwise_dfs = []

#                 if metric in ['gc', 'granger', 'cacoh']:
#                     # no pairs here: computed over whole multivariate state space 
#                     npairs=1
#                 else:
                
                npairs = len(seed_to_target[0])
                    
                for pair in range(npairs): 
                    if epochs_reref.ch_names[seed_to_target[0][pair]] == epochs_reref.ch_names[seed_to_target[1][pair]]:
                        # skip any same-channel connectivity
                        continue
                    pwise_df = pd.DataFrame(columns=['participant', 'roi1', 'roi2', 'hemi', 'pair_label', 'metric', 'event', 'conn'])
                    pwise_df['conn'] = pwise[:, pair] 
                    pwise_df['participant'] = subj_id
                    pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                    pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                    pwise_df['roi1'] = source_region
                    pwise_df['roi2'] = target_region
                    pwise_df['hemi'] = hemi 
                    pwise_df['metric'] = metric
                    pwise_df['event'] = event
                    pwise_df['band0'] = band1
                    pwise_df['band1'] =  band2
                    
#                     if metric=='granger':
#                         pwise_df['pair_label'] = f'{source_region}:{target_region}'
#                     else:
                    pwise_df['pair_label'] = f'{epochs_reref.ch_names[seed_to_target[0][pair]]}:{epochs_reref.ch_names[seed_to_target[1][pair]]}'
                    # pwise_df['trials'] = np.arange(1, pwise.shape[0]+1)
                    if event in ['gamble_start', 'feedback_start', 'baseline_start']:
                        pwise_df['trials'] = np.arange(1, learn_df[learn_df.participant==subj_id].shape[0]+1)
                        pwise_df = pwise_df.merge(learn_df[(learn_df.participant==subj_id)], on=['participant', 'trials'])
                    # elif event in ['cue_on', 'recog_time']:
                    #     pwise_df['trials_mem'] = combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_mem').reset_index(drop=True).trials_mem.values
                    #     pwise_df = pwise_df.merge(combined_df[(combined_df.participant==subj_id) & (combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by='trials_mem').reset_index(drop=True), 
                    #                               on=['participant', 'trials_mem'])

                    # pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
                    # pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

                    # aggregate
                    pwise_dfs.append(pwise_df)
                pwise_dfs = pd.concat(pwise_dfs)
                conn_group_data.append(pwise_dfs)

    all_pairs_df = pd.concat(conn_group_data)
    all_pairs_df.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df', index=False)



In [ ]:
sns.lmplot(data=all_pairs_df, x='rpe', y='conn', scatter=False, hue='participant', ci=None, 
           legend=False)